# Lab 5: "Shout STOP!" — keyword spotting for a voice emergency stop

**Aim:** Train a small DS-CNN in Google Colab that detects the word **"stop"** (and ignores every other word and machine noise), using the Google *Mini Speech Commands* dataset and MFCC features. Then convert the model to **int8 TensorFlow Lite** so it can run on a microcontroller (model must be under 100 KB).

**Workflow of the problem**

```
Mini Speech Commands (8 words)
        |
        v
Build 3 classes:  stop | _unknown_ | _silence_
        |
        v
Split 80/10/10 BY SPEAKER (test = voices the model never heard)
        |
        v
Load audio (16 kHz, mono, exactly 1 s)
        |
        v
MFCC features  (30 ms window, 20 ms hop  ->  49 x 10)
        |
        v
Augmentation on training data (time shift + background noise)
        |
        v
DS-CNN  ->  train  ->  safety check (misses / false alarms)
        |
        v
int8 TFLite  ->  kws_int8.tflite + kws_config.json  (for Lab 6)
```

## Section 0 — Setup

Import the libraries, fix the random seeds (so results can be repeated) and define the audio settings once. The same settings must be used again in Lab 6.

In [ ]:
# libraries
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm

# fix the random seeds so the results are repeatable
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# audio settings (Lab 6 must use exactly the same values)
SAMPLE_RATE = 16000      # 16 kHz
CLIP_SAMPLES = 16000     # exactly 1 second of audio
WIN_LENGTH = 480         # 30 ms window = 0.030 * 16000 samples
HOP_LENGTH = 320         # 20 ms hop    = 0.020 * 16000 samples
N_FFT = 512              # FFT size (next power of 2 above 480)
N_MELS = 40              # number of mel filters
N_MFCC = 10              # number of MFCC coefficients we keep
LABELS = ["stop", "_unknown_", "_silence_"]   # class 0, 1, 2

print("TensorFlow version:", tf.__version__)

## Section 1 — Download the dataset

Mini Speech Commands has 8 words (down, go, left, no, right, stop, up, yes) with about 1,000 one-second clips each. We download and unzip it, then count the clips in every folder.

In [ ]:
import zipfile

DATA_DIR = "data/mini_speech_commands"
ZIP_PATH = "data/mini_speech_commands.zip"
URL = "http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip"

# download and unzip only if the folder is not there yet
if not os.path.exists(DATA_DIR):
    # step 1: download the zip file
    # (we do not use extract=True, because new Keras versions put the extracted files
    #  in a different folder, "..._extracted", and then DATA_DIR is not found)
    tf.keras.utils.get_file("mini_speech_commands.zip", origin=URL,
                            cache_dir=".", cache_subdir="data")
    # step 2: unzip it ourselves into the "data" folder
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("data")

# every word is a folder (ignore folders that start with "_", like _background_noise_)
words = sorted([f for f in os.listdir(DATA_DIR)
                if os.path.isdir(os.path.join(DATA_DIR, f)) and not f.startswith("_")])
print("Words found:", words)

# count the clips in each folder
for w in words:
    n_clips = len([f for f in os.listdir(os.path.join(DATA_DIR, w)) if f.endswith(".wav")])
    print(w, ":", n_clips, "clips")

## Section 2 — Build the 3 classes (Task 1)

The belt must react only to "stop", so we make three balanced classes:

* **stop** – all clips from the `stop` folder
* **_unknown_** – random clips from the other 7 words (same total as stop, equal share per word)
* **_silence_** – 1-second background-noise clips (same total as stop)

The mini dataset usually has **no** `_background_noise_` folder. So the code uses the real folder if it exists, otherwise it **generates** noise with NumPy (white, pink, brown and a 50/100/150 Hz "machine hum"). The same noise is also used later for augmentation.

In [ ]:
# ----- noise helpers -----
NOISE_DIR = os.path.join(DATA_DIR, "_background_noise_")

# if the real noise folder exists, load those long recordings once
long_noises = []
if os.path.exists(NOISE_DIR):
    for f in os.listdir(NOISE_DIR):
        if f.endswith(".wav"):
            audio, _ = librosa.load(os.path.join(NOISE_DIR, f), sr=SAMPLE_RATE)
            long_noises.append(audio)
print("Real background noise files found:", len(long_noises))


def make_noise(kind):
    # make one generated noise clip of 1 second
    white = np.random.randn(CLIP_SAMPLES)
    if kind == "white":
        sig = white
    elif kind == "pink":
        # pink noise: weaken the high frequencies (spectrum falls like 1/sqrt(f))
        spec = np.fft.rfft(white)
        freqs = np.arange(1, len(spec) + 1)
        sig = np.fft.irfft(spec / np.sqrt(freqs), n=CLIP_SAMPLES)
    elif kind == "brown":
        # brown noise: add up the white noise (even more low-frequency energy)
        sig = np.cumsum(white)
        sig = sig - np.mean(sig)
    else:
        # machine hum: 50, 100 and 150 Hz sine waves + a little white noise
        t = np.arange(CLIP_SAMPLES) / SAMPLE_RATE
        sig = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 100 * t) + 0.3 * np.sin(2 * np.pi * 150 * t)
        sig = sig + 0.1 * white
    return sig / np.max(np.abs(sig))      # make the peak value 1


def get_noise_clips(n):
    # return a list of n noise clips (peak = 1)
    kinds = ["white", "pink", "brown", "hum"]
    clips = []
    for i in range(n):
        if len(long_noises) > 0:
            # real noise: cut a random 1 s piece
            long_audio = random.choice(long_noises)
            start = np.random.randint(0, len(long_audio) - CLIP_SAMPLES)
            clip = long_audio[start:start + CLIP_SAMPLES]
            clip = clip / (np.max(np.abs(clip)) + 1e-9)
        else:
            clip = make_noise(kinds[i % 4])
        clips.append(clip)
    return clips


# noise bank used later for augmentation
noise_bank = get_noise_clips(100)
print("Noise bank size:", len(noise_bank))

In [ ]:
# ----- class 0: stop (all clips) -----
stop_files = sorted([os.path.join(DATA_DIR, "stop", f)
                     for f in os.listdir(os.path.join(DATA_DIR, "stop")) if f.endswith(".wav")])
n_stop = len(stop_files)
print("stop clips:", n_stop)

# ----- class 1: _unknown_ (equal share from the other 7 words) -----
other_words = [w for w in words if w != "stop"]
per_word = n_stop // len(other_words)       # clips taken from each word
extra = n_stop % len(other_words)           # leftover clips, one extra for the first few words

unknown_files = []
for i, w in enumerate(other_words):
    all_files = sorted([f for f in os.listdir(os.path.join(DATA_DIR, w)) if f.endswith(".wav")])
    count = per_word + (1 if i < extra else 0)
    chosen = np.random.choice(all_files, count, replace=False)
    for f in chosen:
        unknown_files.append(os.path.join(DATA_DIR, w, f))
print("_unknown_ clips:", len(unknown_files))

# ----- class 2: _silence_ (noise clips saved as wav files, random low volume) -----
os.makedirs("silence_clips", exist_ok=True)
silence_clips = get_noise_clips(n_stop)
silence_files = []
for i, clip in enumerate(silence_clips):
    volume = np.random.uniform(0.01, 0.3)                 # random low volume
    path = "silence_clips/silence_" + str(i) + ".wav"
    sf.write(path, clip * volume, SAMPLE_RATE, subtype="PCM_16")
    silence_files.append(path)
print("_silence_ clips:", len(silence_files))

In [ ]:
# put everything in one table (DataFrame): file path + class label
rows = []
for p in stop_files:
    rows.append([p, 0, "stop"])
for p in unknown_files:
    rows.append([p, 1, os.path.basename(os.path.dirname(p))])   # word folder name
for p in silence_files:
    rows.append([p, 2, "noise"])

df = pd.DataFrame(rows, columns=["path", "label", "word"])
df["label_name"] = df["label"].map(lambda x: LABELS[x])
print(df["label_name"].value_counts())
df.head()

## Section 3 — Speaker-wise 80/10/10 split (Task 2)

New workers join every month, so the test set must contain **voices the model has never heard**. The speaker ID is the first part of the file name (`<speakerID>_nohash_<n>.wav`). We shuffle the speakers and give 80% to train, 10% to validation and 10% to test. A speaker never appears in two splits.

The silence clips have no speaker, so we give each split the same number of silence clips as it has stop clips (this keeps the classes balanced).

In [ ]:
# speaker id = text before "_nohash_" in the file name
def get_speaker(path):
    return os.path.basename(path).split("_nohash_")[0]

df["speaker"] = df["path"].apply(get_speaker)
df.loc[df["label"] == 2, "speaker"] = "noise"        # silence has no speaker

# shuffle the unique speakers and cut them 80 / 10 / 10
speakers = np.array(sorted(df[df["label"] != 2]["speaker"].unique()))
np.random.shuffle(speakers)
n_spk = len(speakers)
n_train = int(0.8 * n_spk)
n_val = int(0.1 * n_spk)
train_spk = speakers[:n_train]
val_spk = speakers[n_train:n_train + n_val]
test_spk = speakers[n_train + n_val:]
print("Speakers -> train:", len(train_spk), " val:", len(val_spk), " test:", len(test_spk))

# give every speech clip the split of its speaker
df["split"] = ""
df.loc[df["speaker"].isin(train_spk), "split"] = "train"
df.loc[df["speaker"].isin(val_spk), "split"] = "val"
df.loc[df["speaker"].isin(test_spk), "split"] = "test"

# silence clips: each split gets as many silence clips as it has stop clips
sil_index = np.array(df[df["label"] == 2].index)     # np.array makes a normal editable copy
np.random.shuffle(sil_index)
start = 0
for s in ["train", "val", "test"]:
    n_s = len(df[(df["label"] == 0) & (df["split"] == s)])
    df.loc[sil_index[start:start + n_s], "split"] = s
    start = start + n_s

# check: no speaker is in two splits
train_set = set(df[(df["split"] == "train") & (df["label"] != 2)]["speaker"])
val_set = set(df[(df["split"] == "val") & (df["label"] != 2)]["speaker"])
test_set = set(df[(df["split"] == "test") & (df["label"] != 2)]["speaker"])
assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0
print("Check passed: no speaker appears in two splits")

# balanced dataset table
table = pd.crosstab(df["split"], df["label_name"])
table = table.loc[["train", "val", "test"], LABELS]
table["total"] = table.sum(axis=1)
print("\nClips per split and class:")
table

## Section 4 — Preprocessing (Task 3)

Every clip is loaded as **16 kHz mono** and forced to be **exactly 1 second (16,000 samples)**: short clips are padded with zeros, long clips are cut.

**Framing:** the audio is cut into small frames with a **30 ms window** (480 samples) that moves forward by a **20 ms hop** (320 samples). For 1 second of audio this gives about `1 + (16000 - 512) // 320 = 49` frames (the FFT size is 512, so each 480-sample window is zero-padded to 512). Each frame becomes 10 MFCC numbers, so one clip becomes a **49 x 10** picture.

In [ ]:
def load_audio(path):
    # load as 16 kHz mono
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    # make it exactly 1 second
    if len(audio) < CLIP_SAMPLES:
        audio = np.pad(audio, (0, CLIP_SAMPLES - len(audio)))   # pad with zeros
    else:
        audio = audio[:CLIP_SAMPLES]                             # cut the extra part
    return audio


# quick test on one clip
test_audio = load_audio(stop_files[0])
print("Audio shape:", test_audio.shape, " -> duration =", len(test_audio) / SAMPLE_RATE, "s")

## Section 5 — Show what the machine "hears" (Task 4)

`get_mfcc` turns a 1-second clip into a 49 x 10 MFCC matrix. Below we plot the **waveform**, **spectrogram** and **MFCC** for one "stop" clip and one "go" clip.

In [ ]:
def get_mfcc(audio):
    # MFCC with 30 ms window and 20 ms hop; center=False gives exactly 49 frames
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT,
                                win_length=WIN_LENGTH, hop_length=HOP_LENGTH,
                                n_mels=N_MELS, center=False)
    return mfcc.T       # shape (frames, coefficients) = (49, 10)


print("MFCC shape:", get_mfcc(test_audio).shape)

In [ ]:
# pick one "stop" clip and one "go" clip from the dataset folders
stop_path = os.path.join(DATA_DIR, "stop", sorted(os.listdir(os.path.join(DATA_DIR, "stop")))[0])
go_path = os.path.join(DATA_DIR, "go", sorted(os.listdir(os.path.join(DATA_DIR, "go")))[0])

fig, axes = plt.subplots(3, 2, figsize=(12, 10))
for col, (name, path) in enumerate([("stop", stop_path), ("go", go_path)]):
    audio = load_audio(path)

    # row 1: waveform
    time = np.arange(len(audio)) / SAMPLE_RATE
    axes[0, col].plot(time, audio)
    axes[0, col].set_title("Waveform - " + name)
    axes[0, col].set_xlabel("Time (s)")
    axes[0, col].set_ylabel("Amplitude")

    # row 2: spectrogram (in dB)
    stft = librosa.stft(audio, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    spec_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
    img1 = librosa.display.specshow(spec_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                                    x_axis="time", y_axis="hz", ax=axes[1, col])
    axes[1, col].set_title("Spectrogram - " + name)
    fig.colorbar(img1, ax=axes[1, col], format="%+2.0f dB")

    # row 3: MFCC (49 frames x 10 coefficients)
    mfcc = get_mfcc(audio)
    img2 = librosa.display.specshow(mfcc.T, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                                    x_axis="time", ax=axes[2, col])
    axes[2, col].set_title("MFCC (49 x 10) - " + name)
    axes[2, col].set_ylabel("MFCC coefficient")
    fig.colorbar(img2, ax=axes[2, col])

plt.tight_layout()
plt.show()

**Why do the MFCC pictures of "stop" and "go" look different? (explanation for the manager)**

1. "stop" starts with a hissy **/s/** (high-frequency energy), then a short **/t/** burst, a vowel, and a **/p/** closure, so its MFCC looks **broken into several different parts** over time.
2. "go" is one smooth **voiced** sound: a short /g/ that flows into a long vowel, so its MFCC looks **smooth and steady**, with most of the energy at low frequencies.
3. The DS-CNN learns these different time-frequency patterns, so it can tell "stop" apart from "go" without ever "understanding" the word.

## Section 6 — Training data with augmentation (Task 6, data part)

Two simple augmentations are applied on the **raw waveform**, for the **training set only**:

* **Time shift** (up to ±100 ms) – for workers who start speaking early or late.
* **Background noise** – a random noise clip is mixed in at a random volume, for a loud factory.

For every training clip we keep the original and add **one** augmented copy. Validation and test data stay clean. Finally, all clips are converted to MFCC and normalised using the **training set** mean and standard deviation only.

In [ ]:
def time_shift(audio):
    # shift the audio by up to +-100 ms (1600 samples)
    max_shift = int(0.1 * SAMPLE_RATE)
    shift = np.random.randint(-max_shift, max_shift + 1)
    shifted = np.roll(audio, shift)
    # the part that wrapped around is set to zero
    if shift > 0:
        shifted[:shift] = 0
    elif shift < 0:
        shifted[shift:] = 0
    return shifted


def add_noise(audio):
    # mix in a random noise clip at a random volume
    noise = random.choice(noise_bank)
    volume = np.random.uniform(0.05, 0.3)
    noisy = audio + volume * noise
    return np.clip(noisy, -1.0, 1.0)


def make_features(split_name, augment):
    # convert all clips of one split into MFCC arrays
    part = df[df["split"] == split_name]
    X_list = []
    y_list = []
    for path, label in tqdm(zip(part["path"], part["label"]), total=len(part), desc=split_name):
        audio = load_audio(path)
        X_list.append(get_mfcc(audio))
        y_list.append(label)
        if augment:
            # one extra augmented copy (shift + noise)
            audio_aug = add_noise(time_shift(audio))
            X_list.append(get_mfcc(audio_aug))
            y_list.append(label)
    X = np.array(X_list, dtype=np.float32)
    X = X[..., np.newaxis]            # add channel axis -> (N, 49, 10, 1)
    y = np.array(y_list)
    return X, y


# this takes a few minutes
X_train, y_train = make_features("train", augment=True)
X_val, y_val = make_features("val", augment=False)
X_test, y_test = make_features("test", augment=False)
print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)

In [ ]:
# normalise with the mean and std of the TRAINING set only
train_mean = float(X_train.mean())
train_std = float(X_train.std())
print("Train mean:", train_mean, " Train std:", train_std)

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

## Section 7 — DS-CNN model (Task 5)

A **DS-CNN** (depthwise separable CNN) is small because each block splits a normal convolution into a cheap depthwise convolution (one 3x3 filter per channel) and a 1x1 pointwise convolution that mixes the channels. The input is the 49 x 10 MFCC and the output is 3 class probabilities.

In [ ]:
# BatchNormalization momentum 0.9 (instead of the default 0.99) lets its running averages
# settle in a few epochs, so the validation accuracy is correct from the start
def bn():
    return layers.BatchNormalization(momentum=0.9)


def build_model():
    model = keras.Sequential()
    model.add(layers.Input(shape=(49, 10, 1)))

    # first normal convolution
    model.add(layers.Conv2D(64, (10, 4), strides=(2, 2), padding="same"))
    model.add(bn())
    model.add(layers.ReLU())

    # 4 depthwise separable blocks
    for i in range(4):
        model.add(layers.DepthwiseConv2D((3, 3), padding="same"))
        model.add(bn())
        model.add(layers.ReLU())
        model.add(layers.Conv2D(64, (1, 1)))
        model.add(bn())
        model.add(layers.ReLU())

    # classifier
    model.add(layers.Dropout(0.3))
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(3, activation="softmax"))
    return model


model = build_model()
model.summary()

total_params = model.count_params()
est_size_kb = total_params * 1 / 1024        # int8 = 1 byte per parameter
print("\nTotal parameters:", total_params)
print("Estimated int8 size: about", round(est_size_kb, 1), "KB")
if est_size_kb < 100:
    print("Fits under the 100 KB limit")
else:
    print("Too big for the 100 KB limit")

## Section 8 — Training (Task 6)

We train with Adam (learning rate 0.001) for up to 30 epochs. **Early stopping** stops training when the validation loss does not improve for 5 epochs and keeps the best weights.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                           restore_best_weights=True)

history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=30, batch_size=64,
                    callbacks=[early_stop], verbose=2)

In [ ]:
# accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.show()

## Section 9 — Safety check (Task 7)

Test accuracy on **unseen speakers**, the confusion matrix, and the two numbers the safety officer wants:

* **Misses** – real "stop" words that the model did *not* catch (dangerous: the belt keeps running)
* **False alarms** – other words or noise that the model wrongly called "stop" (costly: the line halts for no reason)

In [ ]:
# predictions on the test set
probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(probs, axis=1)

test_acc = np.mean(y_pred == y_test) * 100
print("Test accuracy (float model):", round(test_acc, 2), "%")

# confusion matrix (rows = true class, columns = predicted class)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (test set)")
plt.show()

print(classification_report(y_test, y_pred, target_names=LABELS, digits=3, zero_division=0))

# the two safety numbers
misses = cm[0, 1] + cm[0, 2]            # true stop, predicted unknown or silence
false_alarms = cm[1, 0] + cm[2, 0]      # true unknown or silence, predicted stop
print("Total stop clips in test set :", cm[0].sum())
print("MISSES (stop not caught)     :", misses)
print("FALSE ALARMS (wrong stop)    :", false_alarms)

**Which is worse?** A **miss** is far worse for safety. If the model misses "stop", the belt keeps running while a hand may be caught in the rollers. A **false alarm** only stops the line for a moment and costs production time. So for an emergency stop we want misses to be as close to **zero** as possible, even if that means a few more false alarms.

## Section 10 — Make it fit the chip: int8 quantisation (Task 8)

A microcontroller works best with 8-bit integers. We convert the model to:

1. a **float32** TFLite model (no optimisation), and
2. a **full-integer int8** TFLite model. Here a small **representative dataset** (200 training samples) lets the converter measure the value ranges and choose the int8 scales.

Then we run both with the TFLite interpreter on the test set and compare size and accuracy.

In [ ]:
# ----- float32 TFLite model -----
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter.convert()
with open("kws_float32.tflite", "wb") as f:
    f.write(tflite_float)

# ----- full integer int8 TFLite model -----
def representative_data():
    # 200 random training samples to calibrate the int8 ranges
    for i in np.random.choice(len(X_train), 200, replace=False):
        yield [X_train[i:i + 1]]


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_int8 = converter.convert()
with open("kws_int8.tflite", "wb") as f:
    f.write(tflite_int8)

print("Saved kws_float32.tflite and kws_int8.tflite")

In [ ]:
def run_tflite(model_path, X):
    # run a .tflite model on X, one sample at a time; returns predictions
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]

    preds = []
    for i in range(len(X)):
        x = X[i:i + 1]
        if input_info["dtype"] == np.int8:
            # int8 model: convert the float input to int8 using scale and zero point
            scale, zero_point = input_info["quantization"]
            x = np.round(x / scale + zero_point)
            x = np.clip(x, -128, 127).astype(np.int8)
        interpreter.set_tensor(input_info["index"], x)
        interpreter.invoke()
        out = interpreter.get_tensor(output_info["index"])
        preds.append(np.argmax(out))
    return np.array(preds)


pred_float = run_tflite("kws_float32.tflite", X_test)
pred_int8 = run_tflite("kws_int8.tflite", X_test)

acc_keras = test_acc
acc_float = np.mean(pred_float == y_test) * 100
acc_int8 = np.mean(pred_int8 == y_test) * 100

# save the Keras model only to measure its file size
model.save("kws_keras.keras")

results = pd.DataFrame({
    "Model": ["Keras float32", "TFLite float32", "TFLite int8"],
    "Size (KB)": [round(os.path.getsize("kws_keras.keras") / 1024, 1),
                  round(os.path.getsize("kws_float32.tflite") / 1024, 1),
                  round(os.path.getsize("kws_int8.tflite") / 1024, 1)],
    "Test accuracy (%)": [round(acc_keras, 2), round(acc_float, 2), round(acc_int8, 2)],
})
results["Accuracy drop (%)"] = (acc_keras - results["Test accuracy (%)"]).round(2)
results

In [ ]:
# misses and false alarms of the int8 model
cm_int8 = confusion_matrix(y_test, pred_int8)
misses_int8 = cm_int8[0, 1] + cm_int8[0, 2]
false_alarms_int8 = cm_int8[1, 0] + cm_int8[2, 0]
print("int8 model -> MISSES:", misses_int8, "  FALSE ALARMS:", false_alarms_int8)

int8_size_kb = os.path.getsize("kws_int8.tflite") / 1024
if int8_size_kb < 100:
    print("int8 model size:", round(int8_size_kb, 1), "KB -> fits under 100 KB")
else:
    print("int8 model size:", round(int8_size_kb, 1), "KB -> TOO BIG for 100 KB")

## Section 11 — Export for Lab 6

Lab 6 needs the int8 model and **exactly the same settings** used here (MFCC settings, label order, normalisation numbers and the int8 scales). They are saved in `kws_config.json`.

In [ ]:
# read the int8 scale and zero point from the model
interpreter = tf.lite.Interpreter(model_path="kws_int8.tflite")
interpreter.allocate_tensors()
in_scale, in_zero = interpreter.get_input_details()[0]["quantization"]
out_scale, out_zero = interpreter.get_output_details()[0]["quantization"]

config = {
    "sample_rate": SAMPLE_RATE,
    "clip_samples": CLIP_SAMPLES,
    "win_length": WIN_LENGTH,
    "hop_length": HOP_LENGTH,
    "n_fft": N_FFT,
    "n_mels": N_MELS,
    "n_mfcc": N_MFCC,
    "mfcc_shape": [49, 10],
    "mfcc_args": {"function": "librosa.feature.mfcc", "center": False,
                  "n_mfcc": N_MFCC, "n_fft": N_FFT, "win_length": WIN_LENGTH,
                  "hop_length": HOP_LENGTH, "n_mels": N_MELS},
    "labels": LABELS,
    "train_mean": train_mean,
    "train_std": train_std,
    "input_shape": [1, 49, 10, 1],
    "input_dtype": "int8",
    "input_scale": float(in_scale),
    "input_zero_point": int(in_zero),
    "output_scale": float(out_scale),
    "output_zero_point": int(out_zero),
}

with open("kws_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))

In [ ]:
# download both files to your computer (works only inside Google Colab)
try:
    from google.colab import files
    files.download("kws_int8.tflite")
    files.download("kws_config.json")
except ImportError:
    print("Not running in Colab - the files are saved in the current folder.")

## Section 12 — Results summary and viva

The cell below prints the final numbers for the lab report. (Expected from the lab sheet: accuracy about 85–95% on unseen speakers, int8 model about 45–50 KB, less than 1% accuracy loss.)

In [ ]:
print("========== LAB 5 SUMMARY ==========")
print("\n1) Dataset (clips per split and class)")
print(table)

print("\n2) Test accuracy on unseen speakers (float model):", round(acc_keras, 2), "%")

print("\n3) Safety numbers (float model)")
print("   Misses       :", misses, "of", cm[0].sum(), "stop clips")
print("   False alarms :", false_alarms)

print("\n4) Float vs int8")
print(results.to_string(index=False))
print("   int8 misses:", misses_int8, " int8 false alarms:", false_alarms_int8)

print("\n5) Model parameters:", total_params)
print("   int8 model size:", round(int8_size_kb, 1), "KB ->",
      "fits under 100 KB" if int8_size_kb < 100 else "does NOT fit under 100 KB")

### Viva preparation

**Q1. Why use MFCC instead of the raw audio?**
Raw audio has 16,000 numbers per second and is hard to learn from. MFCC keeps only the important shape of the sound spectrum (in a way similar to human hearing) in 49 x 10 = 490 numbers, so the model can be much smaller.

**Q2. Why are depthwise separable convolutions small?**
A normal 3x3 convolution with 64 input and 64 output channels needs 3x3x64x64 = 36,864 weights. A depthwise 3x3 (3x3x64 = 576) plus a 1x1 pointwise (64x64 = 4,096) needs only about 4,700 weights, nearly 8 times fewer, with similar accuracy.

**Q3. Why split by speaker?**
If the same person is in both train and test, the model can memorise that voice and the accuracy looks better than it really is. A speaker-wise split tests on voices the model has never heard, like new workers joining the factory.

**Q4. Why do we need the `_unknown_` and `_silence_` classes?**
In the factory the model hears many other words ("go", "no", "yes") and machine noise all day. Without these classes the model would be forced to call every sound "stop". With them it learns that "stop" is only one special sound.

**Q5. Why full int8 quantisation for a microcontroller?**
Many microcontrollers have no floating-point unit and very little memory. int8 weights are 4 times smaller than float32 and integer maths is faster and uses less power. Full int8 (inputs, weights and activations) means the chip never needs to do float calculations.

**Q6. What does the representative dataset do?**
For full int8 the converter must know the range of the activation values inside the network. It runs about 200 real samples through the model, records the min/max values, and uses them to calculate the int8 scale and zero point. Without it the ranges would be guessed and the accuracy could fall.

**Q7. Misses vs false alarms — which matters more here?**
Misses. A missed "stop" can injure a worker, while a false alarm only costs some production time. That is why we check misses first, and why in Lab 6 a lower threshold (fewer misses, more false alarms) is the safer choice.

**Q8. Why augment with time shift and background noise?**
Workers do not say "stop" at exactly the same moment in the 1-second window, and the factory is noisy. Shifting and adding noise to the training clips makes the model robust to both, without recording new data.